# Perbaikan Forecast Recursive (Hybrid) — Stabil

**Jalankan setelah Section 0–6 notebook `lstm_harian_hybrid_residual.ipynb`** (model, scaler, dfa, dll. harus sudah ada di kernel).

Masalah lama: log-return di-*compound* tiap langkah → bias negatif kecil menumpuk secara perkalian → revenue meluruh ke ~Rp 22 rb/hari.

Tiga pengaman:
1. **Clip** log-return ke persentil 10–90 data train.
2. **Damp** (φ=0.7): perubahan diredam menuju 0 seiring horizon → revert ke persistensi.
3. **Safety rail**: revenue dijepit ke rentang hari-aktif historis.

## Forecast Hybrid — versi stabil

In [ ]:
# Asumsi tersedia di kernel: model, scaler, dfa, FEATURES, LOOK_BACK, inv_logret, df_tr
# ── Statistik dari data train (untuk clip & rail) ────────────────────────────
_train_logret = dfa['logret'].iloc[1:len(df_tr)].values          # logret in-sample
LR_LO, LR_HI  = np.percentile(_train_logret, [10, 90])
_act_rev      = dfa['revenue'].values
REV_FLOOR, REV_CEIL = np.percentile(_act_rev, 2), _act_rev.max()
print(f'Clip log-return : [{LR_LO:+.3f}, {LR_HI:+.3f}]')
print(f'Rail revenue    : [Rp {REV_FLOOR:,.0f}, Rp {REV_CEIL:,.0f}]')

def forecast_hybrid_stable(model, dfa, scaler, features, look_back, n_steps,
                           phi=0.7, lr_lo=LR_LO, lr_hi=LR_HI,
                           rev_floor=REV_FLOOR, rev_ceil=REV_CEIL):
    window   = scaler.transform(dfa.tail(look_back)[features].values)
    prev_rev = float(dfa['revenue'].iloc[-1])
    last_date = pd.Timestamp(dfa['date'].max())
    rows, d, step = [], last_date, 0
    while len(rows) < n_steps:
        d += pd.Timedelta(days=1)
        if d.weekday() >= 5 or d.month in [1, 7]:
            continue
        step += 1
        raw_lr  = float(inv_logret(model.predict(window[np.newaxis], verbose=0)[0, 0]))
        lr_clip = float(np.clip(raw_lr, lr_lo, lr_hi))       # 1) clip
        lr_damp = lr_clip * (phi ** (step - 1))              # 2) damp menuju 0
        rev     = float(np.expm1(np.log1p(prev_rev) + lr_damp))
        rev     = float(np.clip(rev, rev_floor, rev_ceil))   # 3) rail
        # susun baris fitur utk langkah berikutnya (logret efektif = lr_damp)
        woy = d.isocalendar().week
        raw = [lr_damp,
               np.sin(2*np.pi*woy/52), np.cos(2*np.pi*woy/52),
               np.sin(2*np.pi*d.month/12), np.cos(2*np.pi*d.month/12),
               np.sin(2*np.pi*d.weekday()/5), np.cos(2*np.pi*d.weekday()/5),
               0]
        window   = np.vstack([window[1:], scaler.transform([raw])[0]])
        prev_rev = rev
        rows.append({'tanggal': d.date(), 'hari': d.strftime('%A'),
                     'prediksi_revenue': int(round(rev))})
    return pd.DataFrame(rows)

fc = forecast_hybrid_stable(model, dfa, scaler, FEATURES, LOOK_BACK, 7)
print('\n=== Prediksi 7 Hari Aktif (Hybrid STABIL) ===')
print(fc.to_string(index=False))
print(f'\nTotal: Rp {fc.prediksi_revenue.sum():,} | Rata-rata: Rp {fc.prediksi_revenue.mean():,.0f}')

## Pembanding: Seasonal-Naive (rekomendasi untuk app)

In [ ]:
# Prediksi tiap hari ke depan = rata-rata K kemunculan terakhir hari-yang-sama
def forecast_seasonal_naive(dfa, n_steps, k=4):
    last_date = pd.Timestamp(dfa['date'].max())
    by_dow = {w: dfa[dfa['day_of_week'] == w]['revenue'].tail(k).mean()
              for w in range(5)}
    rows, d = [], last_date
    while len(rows) < n_steps:
        d += pd.Timedelta(days=1)
        if d.weekday() >= 5 or d.month in [1, 7]:
            continue
        rows.append({'tanggal': d.date(), 'hari': d.strftime('%A'),
                     'prediksi_revenue': int(round(by_dow[d.weekday()]))})
    return pd.DataFrame(rows)

fc_sn = forecast_seasonal_naive(dfa, 7, k=4)
print('=== Prediksi 7 Hari Aktif (Seasonal-Naive, k=4) ===')
print(fc_sn.to_string(index=False))
print(f'\nTotal: Rp {fc_sn.prediksi_revenue.sum():,} | Rata-rata: Rp {fc_sn.prediksi_revenue.mean():,.0f}')

## Visualisasi: 30 hari historis + dua forecast

In [ ]:
last30 = dfa.tail(30)[['date','revenue']].copy()
fc_d  = pd.to_datetime(fc['tanggal']);     fc_sn_d = pd.to_datetime(fc_sn['tanggal'])

fig, ax = plt.subplots(figsize=(14,5))
ax.plot(last30['date'], last30['revenue'], 'o-', color='steelblue', label='Historis (30 hari)')
ax.plot(fc_d,   fc['prediksi_revenue'],   's--', color='crimson', label='LSTM Hybrid (stabil)')
ax.plot(fc_sn_d, fc_sn['prediksi_revenue'],'^--', color='seagreen', label='Seasonal-Naive')
ax.axhline(dfa['revenue'].mean(), color='gray', ls=':', alpha=.6, label='Rata-rata historis')
ax.set_title('Forecast 7 Hari Aktif ke Depan'); ax.legend(); ax.set_ylabel('Revenue (Rp)')
plt.tight_layout(); plt.show()